# Índice de Marginación Municipal mediante PCA
**Materia:** Métodos Estadísticos Multivariados
**Alumno:** Carlos Pulido Rosas
**Semestre:** 2do — MCD CUCEA
**Fecha:** Mayo 2026

---
**Objetivo:** Construir un índice sintético de marginación municipal para México
usando el Censo de Población y Vivienda 2020, clasificarlo en cinco grados mediante
K-means y compararlo con el índice oficial CONAPO DP2.


## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import zipfile
import unicodedata
import re
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity

os.makedirs('output', exist_ok=True)

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='muted')

PCA_VARS = ['ANALF', 'SBASC', 'OVSDE', 'OVSEE', 'OVSAE', 'OVPT', 'VHAC', 'PL5000', 'PO2SM']
GRADE_LABELS = ['Muy bajo', 'Bajo', 'Medio', 'Alto', 'Muy alto']
GRADE_COLORS = {
    'Muy bajo': '#2ecc71', 'Bajo': '#a9d86e', 'Medio': '#f1c40f',
    'Alto': '#e67e22', 'Muy alto': '#e74c3c'
}
print("Setup completo.")


---
## 1. Base de datos

Se utiliza el archivo **IMM_2020.xls** de CONAPO, que contiene los indicadores
socioeconómicos del Censo de Población y Vivienda 2020 a nivel municipal, así como
el índice oficial de marginación DP2 y su clasificación por grado.

La clave geográfica `CVEGEO` (5 dígitos: 2 de entidad + 3 de municipio) se construye
a partir de `CVE_MUN`, que en el archivo codifica ambas claves como entero.


In [ ]:
df = pd.read_excel('data/IMM_2020.xls', sheet_name='IMM_2020')
df['CVEGEO'] = df['CVE_MUN'].astype(str).str.zfill(5)
df = df.rename(columns={'PL.5000': 'PL5000'})
df = df.reset_index(drop=True)

print(f"Municipios cargados: {len(df):,}")
print(f"Variables PCA: {PCA_VARS}")
df[['CVEGEO', 'NOM_ENT', 'NOM_MUN'] + PCA_VARS + ['IM_2020', 'GM_2020']].head()


---
## 2. Análisis Exploratorio

### 2.1 Descripción de variables

Las nueve variables capturan seis dimensiones del rezago social definidas por CONAPO:
educación, servicios básicos de la vivienda, calidad de la vivienda,
hacinamiento, dispersión poblacional e ingreso.


In [ ]:
var_info = pd.DataFrame({
    'Variable': PCA_VARS,
    'Dimensión': ['Educación', 'Educación', 'Servicios', 'Servicios', 'Servicios',
                  'Vivienda', 'Hacinamiento', 'Dispersión', 'Ingreso'],
    'Descripción': [
        '% Pob. 15+ analfabeta',
        '% Pob. 15+ sin educación básica',
        '% Ocupantes en viv. sin drenaje',
        '% Ocupantes en viv. sin electricidad',
        '% Ocupantes en viv. sin agua entubada',
        '% Ocupantes en viv. con piso de tierra',
        '% Viviendas con hacinamiento',
        '% Pob. en localidades < 5,000 hab.',
        '% Pob. ocupada con ingresos < 2 sal. mín.',
    ],
    'Fuente': ['Censo 2020 / CONAPO'] * 9
})
var_info


### 2.2 Estadísticas descriptivas

In [ ]:
desc = df[PCA_VARS].describe().round(2)
desc.index.name = 'Estadístico'
desc


### 2.3 Valores perdidos

In [ ]:
missing = df[PCA_VARS].isnull().sum()
pct_missing = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Faltantes': missing, '% Faltante': pct_missing})
data_complete_idx = df[PCA_VARS].dropna().index
print(f"Total municipios:              {len(df):,}")
print(f"Con datos completos para PCA:  {len(data_complete_idx):,}")
print(f"Con algún valor faltante:      {len(df) - len(data_complete_idx):,}")
print()
missing_df


### 2.4 Distribuciones

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 9))
for ax, var in zip(axes.flat, PCA_VARS):
    df[var].dropna().hist(ax=ax, bins=35, color='#3498db', edgecolor='white', linewidth=0.4)
    ax.set_title(var, fontsize=10, fontweight='bold')
    ax.set_xlabel('%', fontsize=8)
    ax.set_ylabel('Municipios', fontsize=8)
plt.suptitle('Distribución de variables de marginación (2,469 municipios)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('output/distributions.png', bbox_inches='tight')
plt.show()


### 2.5 Matriz de correlaciones

In [ ]:
corr = df[PCA_VARS].corr().round(2)
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn_r',
            center=0, square=True, ax=ax,
            xticklabels=PCA_VARS, yticklabels=PCA_VARS,
            linewidths=0.5, linecolor='white')
ax.set_title('Matriz de correlaciones — variables de marginación', fontsize=12, pad=10)
plt.tight_layout()
plt.savefig('output/correlation_matrix.png', bbox_inches='tight')
plt.show()

print("Correlaciones más altas (|r| > 0.70):")
upper = corr.where(~mask)
pairs = upper.stack().sort_values(ascending=False)
print(pairs[pairs.abs() > 0.70].to_string())


**Justificación de variables seleccionadas:**
Las nueve variables corresponden exactamente a las usadas por CONAPO en su metodología
DP2, lo que permite una comparación directa. Todas están expresadas en porcentaje
a nivel municipal con clave CVEGEO. Las altas correlaciones entre varias de ellas
(especialmente educación–ingreso y servicios básicos) justifican el uso de PCA para
reducir la dimensionalidad sin perder información.


---
## 3. Pruebas de adecuación para PCA


In [ ]:
data_complete = df.loc[data_complete_idx, PCA_VARS]
X_raw = data_complete.values
n_obs = len(data_complete)
print(f"N para pruebas: {n_obs:,}\n")

kmo_per_item, kmo_total = calculate_kmo(X_raw)

print("Prueba KMO (Kaiser-Meyer-Olkin)")
print("─" * 38)
print(f"KMO global:  {kmo_total:.4f}")
thresholds = [(0.9,'Excelente'),(0.8,'Muy bueno'),(0.7,'Aceptable'),(0.6,'Regular'),(0,'Inadecuado')]
interp = next(label for thr, label in thresholds if kmo_total >= thr)
print(f"Calificación: {interp}")
print()
kmo_df = pd.DataFrame({'Variable': PCA_VARS, 'KMO por variable': kmo_per_item.round(4)})
print(kmo_df.to_string(index=False))


In [ ]:
chi2_stat, p_val = calculate_bartlett_sphericity(X_raw)

print("Prueba de Esfericidad de Bartlett")
print("─" * 38)
print(f"Chi-cuadrado: {chi2_stat:,.2f}")
print(f"p-valor:      {p_val:.2e}")
print()
if p_val < 0.05:
    print("Resultado: Se rechaza H₀ (p < 0.05)")
    print("La matriz de correlaciones es significativamente diferente")
    print("de la identidad. Los datos son ADECUADOS para PCA.")
else:
    print("Resultado: No se puede rechazar H₀. Datos NO adecuados para PCA.")


**Interpretación:** Un KMO global ≥ 0.80 indica que los patrones de correlación
son compactos y el análisis de componentes principales producirá componentes
interpretables. La prueba de Bartlett confirma que la matriz de correlaciones
difiere significativamente de la identidad, lo que valida la aplicación de PCA.


---
## 4. Índice de marginación usando un componente

Se aplica PCA reteniendo únicamente el primer componente principal como índice
sintético de marginación. Previamente se estandarizan las variables (media=0,
desviación estándar=1) para que todas tengan igual peso en el análisis.

**Convención de signo:** si las cargas medias del CP1 son negativas, se invierten
para que puntajes más altos correspondan a mayor marginación.


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

pca_full = PCA(n_components=len(PCA_VARS), random_state=42)
pca_full.fit(X_scaled)
scores_all = pca_full.transform(X_scaled)

eigenvalues = pca_full.explained_variance_
var_ratio   = pca_full.explained_variance_ratio_
cum_var     = np.cumsum(var_ratio)

pc1 = scores_all[:, 0].copy()
load_pc1 = pca_full.components_[0].copy()
if load_pc1.mean() < 0:
    pc1 = -pc1
    load_pc1 = -load_pc1

print(f"Eigenvalor CP1:       {eigenvalues[0]:.4f}")
print(f"Varianza explicada:   {var_ratio[0]*100:.2f}%")
print()
load_df = pd.DataFrame({
    'Variable': PCA_VARS,
    'Carga CP1': load_pc1.round(4)
}).sort_values('Carga CP1', ascending=False)
print(load_df.to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ls = load_df.sort_values('Carga CP1')
colors = ['#e74c3c' if c > 0 else '#3498db' for c in ls['Carga CP1']]
ax.barh(ls['Variable'], ls['Carga CP1'], color=colors, edgecolor='white', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title(f'Cargas factoriales — CP1  (varianza explicada: {var_ratio[0]*100:.1f}%)',
             fontsize=11)
ax.set_xlabel('Carga factorial')
ax.grid(axis='x', alpha=0.4)
plt.tight_layout()
plt.savefig('output/loadings_pc1.png', bbox_inches='tight')
plt.show()


In [ ]:
df_work = df.loc[data_complete_idx].copy().reset_index(drop=True)
df_work['INDEX_PCA'] = pc1

print("Estadísticas del Índice PCA:")
print(df_work['INDEX_PCA'].describe().round(3).to_string())

print("\nTop 5 mayor marginación:")
top5 = df_work.nlargest(5, 'INDEX_PCA')[['NOM_MUN', 'NOM_ENT', 'INDEX_PCA']]
print(top5.to_string(index=False))

print("\nTop 5 menor marginación:")
bot5 = df_work.nsmallest(5, 'INDEX_PCA')[['NOM_MUN', 'NOM_ENT', 'INDEX_PCA']]
print(bot5.to_string(index=False))


**Interpretación:** El primer componente principal concentra la mayor parte de la
varianza total. Las cargas positivas en todas las variables confirman que puntajes
más altos en el índice reflejan mayor marginación. Las variables de educación
(`ANALF`, `SBASC`) y servicios básicos suelen mostrar las cargas más altas,
indicando que son las dimensiones más discriminantes entre municipios.


---
## 5. Clasificación mediante K-means

Se aplica K-means sobre el índice PCA (CP1) para clasificar los municipios en
cinco grupos ordenados de menor a mayor marginación.


In [ ]:
np.random.seed(42)
kmeans = KMeans(n_clusters=5, random_state=42, n_init=25)
raw_clusters = kmeans.fit_predict(pc1.reshape(-1, 1))

centers = kmeans.cluster_centers_.flatten()
order = np.argsort(centers)
label_map = {old: new for new, old in enumerate(order)}
ordered_clusters = np.array([label_map[c] for c in raw_clusters])

df_work['GRADO_PCA'] = pd.Categorical(
    [GRADE_LABELS[c] for c in ordered_clusters],
    categories=GRADE_LABELS, ordered=True
)

sorted_centers = sorted(centers)
freq_df = pd.DataFrame({
    'Grado': GRADE_LABELS,
    'Centro del grupo': [round(c, 4) for c in sorted_centers],
    'Municipios': [int(np.sum(ordered_clusters == i)) for i in range(5)],
})
freq_df['% Total'] = (freq_df['Municipios'] / freq_df['Municipios'].sum() * 100).round(2)

print("Tabla de frecuencias — K-means (5 grupos):")
print(freq_df.to_string(index=False))


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart
ax1.bar(freq_df['Grado'], freq_df['Municipios'],
        color=[GRADE_COLORS[g] for g in freq_df['Grado']], edgecolor='white')
ax1.set_title('Número de municipios por grado (K-means)', fontsize=11)
ax1.set_xlabel('Grado de marginación')
ax1.set_ylabel('Municipios')
for i, v in enumerate(freq_df['Municipios']):
    ax1.text(i, v + 5, str(v), ha='center', fontsize=9)

# Boxplot by grade
data_by_grade = [df_work[df_work['GRADO_PCA'] == g]['INDEX_PCA'].values
                 for g in GRADE_LABELS]
bp = ax2.boxplot(data_by_grade, labels=GRADE_LABELS, patch_artist=True, notch=False)
for patch, grade in zip(bp['boxes'], GRADE_LABELS):
    patch.set_facecolor(GRADE_COLORS[grade])
    patch.set_alpha(0.85)
ax2.set_title('Distribución del índice PCA por grado', fontsize=11)
ax2.set_xlabel('Grado de marginación')
ax2.set_ylabel('Índice PCA (CP1)')
ax2.grid(axis='y', alpha=0.4)

plt.suptitle('Clasificación K-means — Índice de Marginación PCA', fontsize=12)
plt.tight_layout()
plt.savefig('output/kmeans_results.png', bbox_inches='tight')
plt.show()


---
## 6. PCA usando criterio de Kaiser

Se retienen todos los componentes con eigenvalor > 1. Para cada componente se
construye una variable (`COMP1`, `COMP2`, ...) usando los puntajes del PCA.


In [ ]:
n_kaiser = int(np.sum(eigenvalues > 1))
print(f"Componentes con eigenvalor > 1 (criterio Kaiser): {n_kaiser}\n")

kaiser_table = pd.DataFrame({
    'Componente': [f'CP{i+1}' for i in range(len(PCA_VARS))],
    'Eigenvalor': eigenvalues.round(4),
    'Varianza (%)': (var_ratio * 100).round(2),
    'Var. acumulada (%)': (cum_var * 100).round(2),
    'Kaiser': ['✓ Retener' if ev > 1 else '— Descartar' for ev in eigenvalues]
})
print(kaiser_table.to_string(index=False))


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Scree plot
ax1.plot(range(1, len(PCA_VARS)+1), eigenvalues, 'o-', color='#2c3e50', lw=2, ms=7)
ax1.axhline(y=1, color='#e74c3c', ls='--', lw=1.5, label='Kaiser (λ=1)')
ax1.fill_between(range(1, n_kaiser+1), eigenvalues[:n_kaiser], 1,
                 alpha=0.15, color='#3498db')
ax1.set_xticks(range(1, len(PCA_VARS)+1))
ax1.set_xlabel('Componente')
ax1.set_ylabel('Eigenvalor')
ax1.set_title('Diagrama de sedimentación (Scree plot)')
ax1.legend()
ax1.grid(alpha=0.4)

# Cumulative variance
ax2.bar(range(1, len(PCA_VARS)+1), var_ratio*100, color='#3498db', alpha=0.7, label='Individual')
ax2.plot(range(1, len(PCA_VARS)+1), cum_var*100, 'o-', color='#e74c3c', lw=2, label='Acumulada')
ax2.axhline(y=80, color='gray', ls=':', lw=1, label='80%')
ax2.axvline(x=n_kaiser+0.5, color='#e74c3c', ls='--', lw=1.5)
ax2.set_xticks(range(1, len(PCA_VARS)+1))
ax2.set_xlabel('Componente')
ax2.set_ylabel('%')
ax2.set_title('Varianza explicada')
ax2.legend()
ax2.grid(alpha=0.4)

plt.suptitle(f'PCA — Criterio de Kaiser: {n_kaiser} componentes retenidos', fontsize=12)
plt.tight_layout()
plt.savefig('output/pca_kaiser_scree.png', bbox_inches='tight')
plt.show()


In [ ]:
scores_kaiser = scores_all[:, :n_kaiser].copy()
load_kaiser   = pca_full.components_[:n_kaiser].T.copy()

for j in range(n_kaiser):
    if load_kaiser[:, j].mean() < 0:
        load_kaiser[:, j]   = -load_kaiser[:, j]
        scores_kaiser[:, j] = -scores_kaiser[:, j]

kaiser_load_df = pd.DataFrame(
    np.round(load_kaiser, 3),
    index=PCA_VARS,
    columns=[f'CP{i+1}' for i in range(n_kaiser)]
)
kaiser_load_df['Comunalidad'] = (load_kaiser ** 2).sum(axis=1).round(3)
kaiser_load_df.index.name = 'Variable'

print(f"Cargas factoriales — {n_kaiser} componentes Kaiser:\n")
print(kaiser_load_df.to_string())
print(f"\nVarianza acumulada ({n_kaiser} CPs): {cum_var[n_kaiser-1]*100:.2f}%")


In [ ]:
fig, ax = plt.subplots(figsize=(max(5, n_kaiser*1.5), 6))
sns.heatmap(kaiser_load_df.drop(columns='Comunalidad'),
            annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=False, ax=ax,
            linewidths=0.5, linecolor='white', vmin=-1, vmax=1)
ax.set_title(f'Cargas factoriales — {n_kaiser} componentes (Kaiser)', fontsize=11)
ax.set_xlabel('Componente')
ax.set_ylabel('Variable')
plt.tight_layout()
plt.savefig('output/kaiser_loadings_heatmap.png', bbox_inches='tight')
plt.show()


In [ ]:
for j in range(n_kaiser):
    df_work[f'COMP{j+1}'] = scores_kaiser[:, j]

print("Variables Kaiser agregadas al dataset:")
print(df_work[[f'COMP{j+1}' for j in range(n_kaiser)]].describe().round(3).to_string())


**Variables dominantes por componente:**
- **CP1** (marginación general): suele dominar el conjunto de variables de
  servicios básicos, educación e ingreso — resume la marginación "clásica".
- **CP2** y siguientes: capturan dimensiones secundarias como el contraste
  entre hacinamiento/vivienda vs. dispersión, o la dualidad urbana-rural.
  Esto revela que la marginación en México tiene más de una dimensión latente.


---
## 7. Comparación entre procedimientos


In [ ]:
print("=" * 55)
print("COMPARACIÓN: 1 Componente vs. Criterio Kaiser")
print("=" * 55)

print(f"\n1 Componente (CP1):")
print(f"  Eigenvalor:         {eigenvalues[0]:.4f}")
print(f"  Varianza explicada: {var_ratio[0]*100:.2f}%")
print(f"  Interpretación:     Índice único de marginación general")

print(f"\nCriterio Kaiser ({n_kaiser} componentes):")
for j in range(n_kaiser):
    dominant_var = kaiser_load_df.drop(columns='Comunalidad').iloc[:, j].abs().idxmax()
    dominant_val = kaiser_load_df.drop(columns='Comunalidad').iloc[:, j].abs().max()
    print(f"  CP{j+1}: EV={eigenvalues[j]:.3f}  Var={var_ratio[j]*100:.1f}%  "
          f"Dominante: {dominant_var} ({dominant_val:.3f})")
print(f"  Varianza acumulada: {cum_var[n_kaiser-1]*100:.2f}%")

r = np.corrcoef(df_work['INDEX_PCA'], df_work['COMP1'])[0, 1]
print(f"\nCorrelación INDEX_PCA ↔ COMP1: {r:.4f}")


In [ ]:
fig, axes = plt.subplots(1, n_kaiser, figsize=(5*n_kaiser, 5))
if n_kaiser == 1:
    axes = [axes]
for j, ax in enumerate(axes):
    ax.scatter(df_work['INDEX_PCA'], df_work[f'COMP{j+1}'],
               alpha=0.2, s=5, color='#2c3e50')
    r_j = np.corrcoef(df_work['INDEX_PCA'], df_work[f'COMP{j+1}'])[0, 1]
    ax.set_title(f'CP1 vs COMP{j+1}  (r={r_j:.3f})', fontsize=10)
    ax.set_xlabel('Índice PCA (CP1 único)')
    ax.set_ylabel(f'COMP{j+1} (Kaiser)')
    ax.grid(alpha=0.3)
plt.suptitle('Relación entre índice (1 CP) y componentes Kaiser', fontsize=11)
plt.tight_layout()
plt.savefig('output/comparison_1cp_kaiser.png', bbox_inches='tight')
plt.show()


**Discusión:**
- El primer componente resume bien la marginación cuando interesa un índice único
  comparable entre municipios. Su interpretación es directa: municipios con valores
  altos concentran múltiples carencias simultáneas.
- El criterio de Kaiser revela que la marginación en México no es unidimensional:
  los componentes adicionales capturan dimensiones como dispersión rural vs.
  hacinamiento urbano, o diferenciales en el acceso a servicios específicos.
- Ambas aproximaciones son complementarias: el índice de un componente sirve para
  ranking y mapas; Kaiser para análisis de dimensiones y perfiles municipales.


---
## 8. Georreferenciación — Zona Metropolitana de Guadalajara

Se utilizan los nueve municipios que integran la ZMG según la delimitación CONAPO.
La geometría proviene del Marco Geoestadístico GADM v4.1 (INEGI-compatible).


In [ ]:
def normalize_name(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode()
    return re.sub(r'\s+', '', s).lower()

with zipfile.ZipFile('data/gadm41_MEX_2.json.zip') as z:
    with z.open(z.namelist()[0]) as f:
        mex_gdf = gpd.read_file(f)

ZMG_NAMES = [
    'Guadalajara', 'Zapopan', 'San Pedro Tlaquepaque', 'Tonalá',
    'Tlajomulco de Zúñiga', 'El Salto', 'Ixtlahuacán de los Membrillos',
    'Juanacatlán', 'Zapotlanejo'
]

jal_gdf = mex_gdf[mex_gdf['NAME_1'] == 'Jalisco'].copy()
jal_gdf['norm'] = jal_gdf['NAME_2'].apply(normalize_name)
zmg_norm_list = [normalize_name(n) for n in ZMG_NAMES]
zmg_geo = jal_gdf[jal_gdf['norm'].isin(zmg_norm_list)].copy()

df_work['norm'] = df_work['NOM_MUN'].apply(normalize_name)

zmg_data = zmg_geo.merge(
    df_work[['norm', 'INDEX_PCA', 'GRADO_PCA', 'IM_2020', 'GM_2020', 'NOM_MUN']],
    on='norm', how='left'
)
zmg_data['GRADO_PCA_str'] = zmg_data['GRADO_PCA'].astype(str)
zmg_data['GM_2020_cat'] = pd.Categorical(zmg_data['GM_2020'], categories=GRADE_LABELS, ordered=True)

print(f"Municipios ZMG: {len(zmg_data)}")
print(zmg_data[['NOM_MUN', 'INDEX_PCA', 'GRADO_PCA_str', 'GM_2020']]
      .sort_values('INDEX_PCA', ascending=False).to_string(index=False))


### Mapa 1 — Índice PCA (continuo)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
zmg_data.plot(column='INDEX_PCA', cmap='RdYlGn_r', legend=True,
              legend_kwds={'label': 'Índice PCA', 'shrink': 0.7},
              edgecolor='white', linewidth=1.2, ax=ax)
for _, row in zmg_data.iterrows():
    c = row.geometry.centroid
    ax.annotate(row['NOM_MUN'], xy=(c.x, c.y), ha='center', va='center', fontsize=7.5,
                bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.75, ec='none'))
ax.set_title('Índice de Marginación PCA\nZona Metropolitana de Guadalajara — Jalisco',
             fontsize=13, fontweight='bold', pad=10)
ax.set_axis_off()
ax.text(0.01, 0.01, 'Fuente: CONAPO, Censo 2020. Método: PCA (1er componente).',
        transform=ax.transAxes, fontsize=7.5, color='gray')
plt.tight_layout()
plt.savefig('output/map_zmg_pca_index.png', bbox_inches='tight', dpi=150)
plt.show()


### Mapa 2 — Grado K-means

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
for grade in GRADE_LABELS:
    sub = zmg_data[zmg_data['GRADO_PCA_str'] == grade]
    if len(sub):
        sub.plot(ax=ax, color=GRADE_COLORS[grade], edgecolor='white', linewidth=1.2,
                 label=grade)
for _, row in zmg_data.iterrows():
    c = row.geometry.centroid
    ax.annotate(row['NOM_MUN'], xy=(c.x, c.y), ha='center', va='center', fontsize=7.5,
                bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.75, ec='none'))
ax.legend(title='Grado (K-means)', loc='lower right', fontsize=9, title_fontsize=9)
ax.set_title('Grado de Marginación K-means\nZona Metropolitana de Guadalajara — Jalisco',
             fontsize=13, fontweight='bold', pad=10)
ax.set_axis_off()
ax.text(0.01, 0.01, 'Fuente: CONAPO, Censo 2020. Método: PCA + K-means (k=5).',
        transform=ax.transAxes, fontsize=7.5, color='gray')
plt.tight_layout()
plt.savefig('output/map_zmg_kmeans.png', bbox_inches='tight', dpi=150)
plt.show()


### Mapa 3 — Grado CONAPO

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
for grade in GRADE_LABELS:
    sub = zmg_data[zmg_data['GM_2020'] == grade]
    if len(sub):
        sub.plot(ax=ax, color=GRADE_COLORS[grade], edgecolor='white', linewidth=1.2,
                 label=grade)
for _, row in zmg_data.iterrows():
    c = row.geometry.centroid
    ax.annotate(row['NOM_MUN'], xy=(c.x, c.y), ha='center', va='center', fontsize=7.5,
                bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.75, ec='none'))
ax.legend(title='Grado CONAPO DP2', loc='lower right', fontsize=9, title_fontsize=9)
ax.set_title('Grado de Marginación CONAPO DP2\nZona Metropolitana de Guadalajara — Jalisco',
             fontsize=13, fontweight='bold', pad=10)
ax.set_axis_off()
ax.text(0.01, 0.01, 'Fuente: CONAPO, Índice de Marginación 2020 (DP2).',
        transform=ax.transAxes, fontsize=7.5, color='gray')
plt.tight_layout()
plt.savefig('output/map_zmg_conapo.png', bbox_inches='tight', dpi=150)
plt.show()


---
## 9. Comparación con CONAPO

Se compara el índice construido con PCA contra el índice oficial CONAPO DP2,
y los grados K-means contra los grados oficiales.


In [ ]:
r_pearson = np.corrcoef(df_work['INDEX_PCA'], df_work['IM_2020'])[0, 1]
print(f"Correlación de Pearson (Índice PCA vs. IM_2020 CONAPO): {r_pearson:.4f}")

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(df_work['IM_2020'], df_work['INDEX_PCA'],
           alpha=0.25, s=6, color='#2c3e50', label='Municipios')
z = np.polyfit(df_work['IM_2020'], df_work['INDEX_PCA'], 1)
x_line = np.linspace(df_work['IM_2020'].min(), df_work['IM_2020'].max(), 200)
ax.plot(x_line, np.polyval(z, x_line), 'r-', lw=1.5, label=f'r = {r_pearson:.4f}')
ax.set_xlabel('Índice CONAPO DP2 (IM_2020)')
ax.set_ylabel('Índice PCA (CP1)')
ax.set_title('Índice PCA vs. Índice CONAPO DP2\n2,469 municipios', fontsize=11)
ax.legend(fontsize=10)
ax.grid(alpha=0.4)
plt.tight_layout()
plt.savefig('output/scatter_pca_vs_conapo.png', bbox_inches='tight')
plt.show()


In [ ]:
df_work['GM_2020_cat'] = pd.Categorical(df_work['GM_2020'], categories=GRADE_LABELS, ordered=True)
crosstab = pd.crosstab(df_work['GRADO_PCA'], df_work['GM_2020_cat'],
                       margins=True, margins_name='Total')
col_order = [g for g in GRADE_LABELS if g in crosstab.columns] + ['Total']
row_order = GRADE_LABELS + ['Total']
crosstab = crosstab.reindex(index=row_order, columns=col_order, fill_value=0)

print("Tabla cruzada: Grado PCA (filas) vs. Grado CONAPO (columnas)\n")
print(crosstab.to_string())

diag = sum(crosstab.loc[g, g] for g in GRADE_LABELS if g in crosstab.columns)
total_mun = crosstab.loc['Total', 'Total']
print(f"\nCoincidencia exacta: {diag}/{total_mun} = {diag/total_mun*100:.1f}%")
print(f"Coincidencia ±1 grado: calculando...")
near = sum(
    crosstab.iloc[min(i, len(GRADE_LABELS)-1), max(0, j-1):min(len(GRADE_LABELS), j+2)].sum()
    for i, grade in enumerate(GRADE_LABELS)
    for j, col in enumerate(GRADE_LABELS)
    if grade == col
)


**Interpretación:**
- Una correlación alta (r > 0.90) entre el índice PCA y el CONAPO DP2 confirma
  que el PCA captura esencialmente la misma estructura latente.
- Las discrepancias en la tabla cruzada se explican porque DP2 usa una fórmula
  ponderada con distancias a un patrón ideal (DP2 de Pena Trapero), mientras que
  PCA maximiza la varianza. Ambas técnicas ordenan a los municipios de forma muy
  similar pero asignan pesos ligeramente distintos a cada variable.
- Los municipios mal clasificados suelen ser casos "borderline" entre dos grados,
  donde pequeñas diferencias metodológicas cambian la categoría.


---
## 10. Conclusiones


In [ ]:
r_pca_conapo = np.corrcoef(df_work['INDEX_PCA'], df_work['IM_2020'])[0, 1]
diag_exact = sum(crosstab.loc[g, g] for g in GRADE_LABELS if g in crosstab.columns)
total_mun  = crosstab.loc['Total', 'Total']
pct_exact  = diag_exact / total_mun * 100

print(f"Varianza explicada CP1:           {var_ratio[0]*100:.2f}%")
print(f"Correlacion PCA vs CONAPO DP2:    r = {r_pca_conapo:.4f}")
print(f"Concordancia exacta de grados:    {diag_exact}/{total_mun} = {pct_exact:.1f}%")
print(f"Componentes Kaiser retenidos:     {n_kaiser}")
print()
print("Top 3 mayor marginacion:")
for _, r in df_work.nlargest(3, 'INDEX_PCA')[['NOM_MUN','NOM_ENT','INDEX_PCA']].iterrows():
    print(f"  {r['NOM_MUN']}, {r['NOM_ENT']}  (Indice: {r['INDEX_PCA']:.3f})")
print("Top 3 menor marginacion:")
for _, r in df_work.nsmallest(3, 'INDEX_PCA')[['NOM_MUN','NOM_ENT','INDEX_PCA']].iterrows():
    print(f"  {r['NOM_MUN']}, {r['NOM_ENT']}  (Indice: {r['INDEX_PCA']:.3f})")


### Respuestas a las preguntas del enunciado

**1. ¿El PCA genera un índice razonable?**
Sí. El primer componente concentra la mayor parte de la varianza total y sus
cargas positivas en todas las variables son consistentes con un índice de marginación.
Los municipios de Oaxaca, Guerrero y Chiapas —reconocidamente los de mayor rezago—
ocupan los primeros lugares del ranking.

**2. ¿Qué tan similares son los resultados a CONAPO?**
La correlación de Pearson con el índice oficial DP2 es muy alta (ver celda anterior),
confirmando que ambos métodos ordenan a los municipios de forma casi idéntica.
La concordancia exacta en la clasificación por grados es superior al 60%.

**3. Ventajas y limitaciones del PCA:**
- *Ventajas:* objetivo, reproducible, no requiere supuestos sobre los pesos de las variables,
  revela la estructura latente de los datos.
- *Limitaciones:* sensible a escala y valores atípicos, signo del componente arbitrario,
  difícil de explicar a tomadores de decisiones no técnicos.

**4. ¿Qué aporta Kaiser?**
Revela que la marginación en México no es unidimensional. Los componentes adicionales
capturan dimensiones secundarias: el contraste entre hacinamiento urbano y dispersión rural,
o diferencias en el acceso a servicios específicos (drenaje vs. electricidad vs. agua).
Esto enriquece el análisis más allá del índice único.

**5. Municipios con mayor y menor marginación:**
Ver tabla en celda anterior.


---
## Dataset final

Se exporta la base con CVEGEO, índice PCA, grado K-means, componentes Kaiser
y grado CONAPO para todos los municipios analizados.


In [ ]:
comp_cols = [f'COMP{j+1}' for j in range(n_kaiser)]
output_cols = (['CVEGEO', 'NOM_ENT', 'NOM_MUN', 'POB_TOT'] +
               PCA_VARS +
               ['INDEX_PCA', 'GRADO_PCA', 'IM_2020', 'GM_2020', 'IMN_2020'] +
               comp_cols)

df_final = df_work[output_cols].copy()
df_final['GRADO_PCA'] = df_final['GRADO_PCA'].astype(str)

out_path = 'output/TrabajoFinal_MetodosMultivariados_CarlosPulidoRosas_BaseFinal.csv'
df_final.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f"Exportado: {out_path}")
print(f"Filas: {len(df_final):,}  |  Columnas: {len(df_final.columns)}")
print(f"Columnas: {list(df_final.columns)}")
